# Manhwa Studio — Google Colab + Google Drive

Painel de edição no Colab inspirado no Manhwa Studio da Lovable. Navegue pelas pastas do Google Drive, vincule um vídeo e uma moldura, confira a prévia, ajuste o enquadramento e renderize com FFmpeg/GPU quando disponível.

**Não baixa vídeos do YouTube.** Os arquivos são selecionados diretamente do Google Drive. Use somente mídia que você tenha direito ou autorização para editar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
WORK = Path('/content/drive/MyDrive/ManhwaStudio')
INPUT = WORK / 'entrada'
OUTPUT = WORK / 'exportados'
INPUT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Entrada:', INPUT)
print('Saída:', OUTPUT)

In [ ]:
# Instalação automática do ambiente e verificação da GPU
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!python -m pip install -q --upgrade pip
!python -m pip install -q ipywidgets pillow imageio-ffmpeg

import subprocess, shutil
assert shutil.which('ffmpeg'), 'FFmpeg não foi instalado corretamente.'
print(subprocess.run(['ffmpeg','-version'],capture_output=True,text=True).stdout.splitlines()[0])
gpu = subprocess.run(['nvidia-smi'],capture_output=True,text=True)
if gpu.returncode == 0:
    print('GPU NVIDIA detectada:')
    print(gpu.stdout[:1800])
else:
    print('GPU NVIDIA não detectada. Selecione Ambiente de execução > Alterar tipo de ambiente de execução > GPU.')
enc = subprocess.run(['ffmpeg','-hide_banner','-encoders'],capture_output=True,text=True).stdout
print('Encoders disponíveis:', ', '.join(x for x in ['h264_nvenc','h264_qsv','h264_amf','libx264'] if x in enc))


In [ ]:
!git clone --depth 1 --branch google-colab-drive https://github.com/manhwavaz1-lab/manhwa-recreator-studio.git /content/manhwa-recreator-studio
%cd /content/manhwa-recreator-studio

## Painel de edição

1. Navegue nas pastas do Google Drive e vincule o vídeo.
2. Navegue novamente e vincule a moldura PNG/JPG/WEBP.
3. Ajuste formato, resolução, zoom e posição; confira as prévias.
4. Clique em **Renderizar vídeo**. O arquivo será salvo em `Meu Drive/ManhwaStudio/exportados`.

A interface usa quatro ações separadas, como no fluxo de edição: **Vincular vídeo**, **Vincular moldura**, **Renderizar vídeo** e **Baixar MP4**.

## Interface do Manhwa Studio

O navegador abaixo lista o conteúdo real do Google Drive já montado no Colab. Selecione uma pasta e um arquivo; não é necessário colar links nem baixar do YouTube.

In [ ]:
import ipywidgets as w
from IPython.display import display, HTML, clear_output, Video, Image
from pathlib import Path
from google.colab import files
import subprocess, sys, mimetypes, html

ROOT = Path('/content/drive/MyDrive')
WORK = ROOT / 'ManhwaStudio'
INPUT = WORK / 'entrada'
OUTPUT = WORK / 'exportados'
INPUT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

state = {'video': None, 'frame': None, 'rendered': None}
current_folder = {'path': INPUT}  # abre diretamente Meu Drive/ManhwaStudio/entrada

title = w.HTML("""
<div style="background:#090b10;color:#fff;padding:22px 24px;border:1px solid #292f3a;border-radius:14px;font-family:Arial">
 <div style="display:flex;align-items:center;gap:12px"><div style="background:#ef3349;color:white;font-size:22px;font-weight:900;padding:10px 14px;border-radius:9px">M</div>
 <div><div style="font-size:18px;font-weight:900;letter-spacing:2px">MANHWA STUDIO</div><div style="color:#929baa;font-size:10px;letter-spacing:2px">VIDEO FRAME CREATOR · GOOGLE DRIVE + GPU</div></div></div>
 <div style="margin-top:18px;color:#f05b6b;font-size:11px;font-weight:bold;letter-spacing:1.5px">PAINEL DE EDIÇÃO</div>
 <div style="font-size:23px;font-weight:800;margin-top:5px">Seu vídeo, com <span style="color:#f04459">sua moldura.</span></div>
 <div style="color:#aab2c0;font-size:12px;margin-top:7px">A pasta de entrada abre automaticamente. Selecione vídeo e moldura, ajuste o enquadramento e renderize com FFmpeg.</div>
</div>
""")

folder_label = w.HTML()
folder_select = w.Dropdown(description='Pasta:', layout=w.Layout(width='100%'), style={'description_width':'70px'})
file_select = w.Dropdown(description='Arquivo:', layout=w.Layout(width='100%'), style={'description_width':'70px'})
refresh_btn = w.Button(description='Atualizar lista', icon='refresh', layout=w.Layout(width='160px'))
up_btn = w.Button(description='Pasta acima', icon='level-up', layout=w.Layout(width='140px'))
home_btn = w.Button(description='Abrir pasta de entrada', icon='home', button_style='info', layout=w.Layout(width='190px'))
browser_status = w.Output()

def list_folders(path):
    try:
        return sorted([p for p in path.iterdir() if p.is_dir() and not p.name.startswith('.')], key=lambda p:p.name.lower())
    except PermissionError:
        return []

def refresh_browser(*_):
    path = current_folder['path']
    folder_label.value = "<div style='color:#f05b6b;font-size:11px;margin:8px 0'>GOOGLE DRIVE / " + html.escape(str(path.relative_to(ROOT)) if path != ROOT else 'Meu Drive') + "</div>"
    dirs = list_folders(path)
    folder_select.options = [(('📁 ' + p.name), str(p)) for p in dirs]
    folder_select.value = None
    try:
        files_here = sorted([p for p in path.iterdir() if p.is_file() and p.suffix.lower() in ('.mp4','.mov','.mkv','.webm','.m4v','.avi','.png','.jpg','.jpeg','.webp')], key=lambda p:p.name.lower())
    except PermissionError:
        files_here = []
    file_select.options = [(('🎬 ' if p.suffix.lower() in ('.mp4','.mov','.mkv','.webm','.m4v','.avi') else '🖼️ ') + p.name, str(p)) for p in files_here]
    file_select.value = None
    with browser_status:
        clear_output()
        print(f'{len(dirs)} pastas e {len(files_here)} arquivos de mídia nesta pasta.')

def open_folder(change):
    if change['new']:
        current_folder['path'] = Path(change['new'])
        refresh_browser()

def go_up(_):
    if current_folder['path'] != ROOT and ROOT in current_folder['path'].parents:
        current_folder['path'] = current_folder['path'].parent
        refresh_browser()

def open_input_folder(_):
    current_folder['path'] = INPUT
    refresh_browser()

folder_select.observe(open_folder, names='value')
refresh_btn.on_click(refresh_browser)
up_btn.on_click(go_up)
home_btn.on_click(open_input_folder)
refresh_browser()

video_status = w.HTML("<span style='color:#9ca3af'>Nenhum vídeo vinculado</span>")
frame_status = w.HTML("<span style='color:#9ca3af'>Nenhuma moldura vinculada</span>")
video_preview = w.Output()
frame_preview = w.Output()

def link_video(_):
    value = file_select.value
    if not value:
        with browser_status:
            print('Selecione um arquivo de vídeo na lista Arquivo.')
        return
    p = Path(value)
    if p.suffix.lower() not in ('.mp4','.mov','.mkv','.webm','.m4v','.avi'):
        with browser_status: print('O arquivo selecionado não é um vídeo.')
        return
    state['video'] = p
    state['rendered'] = None
    video_status.value = "<span style='color:#4ade80'>✓ " + html.escape(p.name) + "</span>"
    with video_preview:
        clear_output()
        print('Prévia do vídeo selecionado:')
        display(Video(filename=str(p), embed=False, width=520))

def link_frame(_):
    value = file_select.value
    if not value:
        with browser_status: print('Selecione uma imagem na lista Arquivo.')
        return
    p = Path(value)
    if p.suffix.lower() not in ('.png','.jpg','.jpeg','.webp'):
        with browser_status: print('O arquivo selecionado não é uma imagem de moldura.')
        return
    state['frame'] = p
    state['rendered'] = None
    frame_status.value = "<span style='color:#4ade80'>✓ " + html.escape(p.name) + "</span>"
    with frame_preview:
        clear_output()
        print('Prévia da moldura:')
        display(Image(filename=str(p), width=300))

link_video_btn = w.Button(description='VINCULAR VÍDEO', button_style='info', icon='link', layout=w.Layout(width='48%',height='42px'))
link_frame_btn = w.Button(description='VINCULAR MOLDURA', button_style='info', icon='image', layout=w.Layout(width='48%',height='42px'))
link_video_btn.on_click(link_video)
link_frame_btn.on_click(link_frame)

format_select = w.ToggleButtons(options=[('Horizontal 16:9','16:9'),('Vertical 9:16','9:16')],value='16:9',description='Formato:')
resolution_select = w.Dropdown(options=[('Full HD 1080p','1080p'),('Ultra HD 4K','4K')],value='1080p',description='Resolução:')
encoder_select = w.Dropdown(options=[('Automático (GPU)','auto'),('NVIDIA NVENC','nvenc'),('CPU x264','cpu')],value='auto',description='Encoder:')
zoom_slider = w.IntSlider(value=100,min=50,max=250,step=1,description='Zoom:',continuous_update=False,style={'description_width':'90px'})
offset_x = w.IntSlider(value=0,min=-50,max=50,description='Posição X:',continuous_update=False,style={'description_width':'90px'})
offset_y = w.IntSlider(value=0,min=-50,max=50,description='Posição Y:',continuous_update=False,style={'description_width':'90px'})
crop_check = w.ToggleButtons(options=[('Recortar para preencher',True),('Mostrar vídeo inteiro',False)],value=True,description='Ajuste:')
blur_check = w.Checkbox(value=False,description='Desfocar área de marca-d’água própria')
blur_x = w.IntSlider(value=80,min=0,max=99,description='Esquerda %')
blur_y = w.IntSlider(value=85,min=0,max=99,description='Topo %')
blur_w = w.IntSlider(value=15,min=1,max=100,description='Largura %')
blur_h = w.IntSlider(value=8,min=1,max=100,description='Altura %')

progress = w.IntProgress(value=0,min=0,max=100,description='Render:',bar_style='info',layout=w.Layout(width='100%'))
render_status = w.Output()
render_btn = w.Button(description='RENDERIZAR VÍDEO',button_style='success',icon='play',layout=w.Layout(width='100%',height='48px'))
download_btn = w.Button(description='BAIXAR MP4',button_style='warning',icon='download',layout=w.Layout(width='100%',height='42px'),disabled=True)

def render_video(_):
    if not state['video'] or not state['frame']:
        with render_status:
            clear_output()
            print('Primeiro vincule um vídeo e uma moldura usando os botões acima.')
        return
    render_btn.disabled = True
    download_btn.disabled = True
    progress.value = 5
    with render_status:
        clear_output()
        try:
            out = OUTPUT / (state['video'].stem + '-manhwa-studio.mp4')
            width_height = ('1920','1080') if format_select.value == '16:9' else ('1080','1920')
            cmd=[sys.executable,'colab/renderer.py','--video',str(state['video']),'--frame',str(state['frame']),
                 '--output',str(out),'--format',format_select.value,'--resolution',resolution_select.value,
                 '--encoder',encoder_select.value,'--fps','30','--zoom',str(zoom_slider.value),
                 '--offset-x',str(offset_x.value),'--offset-y',str(offset_y.value)]
            if crop_check.value: cmd.append('--crop')
            if blur_check.value:
                cmd += ['--blur',','.join(map(str,[blur_x.value,blur_y.value,blur_w.value,blur_h.value]))]
            print('Renderizando com FFmpeg. A GPU será usada se o encoder estiver disponível...')
            progress.value = 20
            result = subprocess.run(cmd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
            print(result.stdout[-9000:])
            if result.returncode != 0:
                raise RuntimeError('FFmpeg retornou erro. Consulte o log acima.')
            state['rendered'] = out
            progress.value = 100
            download_btn.disabled = False
            print('Concluído! Arquivo salvo em:',out)
        except Exception as e:
            progress.bar_style='danger'
            print('ERRO:',str(e))
        finally:
            render_btn.disabled = False

def download_mp4(_):
    if state['rendered'] and state['rendered'].exists():
        files.download(str(state['rendered']))
    else:
        with render_status: print('Renderize o vídeo antes de baixar.')

render_btn.on_click(render_video)
download_btn.on_click(download_mp4)

panel_style = """
<style>
.jp-OutputArea-output{background:#0b0d12!important}
.widget-label{color:#d1d5db!important}
.widget-button{border-radius:8px!important;font-weight:700!important}
.widget-dropdown select,.widget-text input{background:#11151d!important;color:#f3f4f6!important}
</style>"""
display(HTML(panel_style))
display(title)
display(w.HTML("<h3 style='color:#f5f5f5'>01 · NAVEGADOR DO GOOGLE DRIVE</h3><p style='color:#9ca3af'>Escolha a pasta. Os arquivos de vídeo e imagem aparecem na lista Arquivo.</p>"))
display(folder_label, w.HBox([home_btn,up_btn,refresh_btn]), folder_select, file_select, browser_status)
display(w.HBox([link_video_btn,link_frame_btn]))
display(w.HTML("<h3 style='color:#f5f5f5'>02 · MÍDIAS VINCULADAS</h3>"))
display(video_status,video_preview,frame_status,frame_preview)
display(w.HTML("<h3 style='color:#f5f5f5'>03 · FORMATO E RESOLUÇÃO</h3>"))
display(format_select,resolution_select,encoder_select)
display(w.HTML("<h3 style='color:#f5f5f5'>04 · AJUSTAR RECORTE E POSIÇÃO</h3>"))
display(crop_check,zoom_slider,offset_x,offset_y)
display(blur_check)
display(w.VBox([blur_x,blur_y,blur_w,blur_h]))
display(w.HTML("<h3 style='color:#f5f5f5'>05 · EXPORTAÇÃO</h3>"))
display(render_btn,progress,download_btn,render_status)
